# Deep Research Agent Walkthrough

This notebook is the hands-on path for the workshop. It builds a practical **LangChain Deep Agents** agent harness backed by an **LLM Wiki** workspace. The goal is not to memorize framework vocabulary. The goal is to see where each piece of the system lives in code, run the deterministic parts, and leave with a repo pattern that can be adapted to real research work.

The default cells do not call a live model or a live AI-Q backend. The optional live run is last and skips itself unless model credentials are configured.

## What you'll build

You will build a small but real research-agent harness: a UV-installable Python package, a `deep-research-agent` console command, deterministic source tools, local skills, subagents, an LLM Wiki workspace, and a final optional live Deep Agents invocation.

The baseline task is to compare current approaches to long-running AI research agents. The task is intentionally general so the group can focus on the harness: how work is planned, where source records live, how skills are loaded, how subagents split work, and how a reviewer inspects the result.

## Setup path

From a fresh checkout, the expected path is:

```bash
uv sync --all-groups
uv run ruff check .
uv run pytest -q
uv run --group notebook jupyter lab notebooks/deep_research_agent_walkthrough.ipynb
```

The notebook writes deterministic example artifacts before any live model call. That gives the workshop a reliable baseline and lets participants inspect the file contract before turning on external systems.

## 1. LangChain Deep Agents framework

Deep Agents is LangChain's higher-level harness for agents that need to do longer, multi-step work. It is built on LangChain and LangGraph. In practice, that means the model is not just receiving a prompt and returning an answer. It is running inside a framework that can manage a task list, call tools, read and write files, delegate to subagents, load skills, apply permissions, stream events, and keep state across a longer run.

For this workshop, read Deep Agents as a software boundary around the model. The model still decides the next step, but the framework decides what capabilities are available, where work can be written, which procedures can be loaded, and how the result can be inspected afterward. That is the difference between a chat transcript and a research workflow.

The core constructor is `create_deep_agent`. A minimal demo can pass only a model, a tool list, and a prompt. A useful research harness usually needs more: a filesystem backend, permission rules, skills, subagents, and a coordinator prompt. Each argument is an engineering decision.

### What each framework piece does

| Piece | Meaning in Deep Agents | Meaning in this repo |
|---|---|---|
| Model | The LLM used by the coordinator and default subagents. | Configured through `DEEP_AGENTS_MODEL`. |
| Tools | Python callables the agent can invoke. | Source records, ledgers, task packets, AI-Q request contracts. |
| System prompt | The coordinator's operating instructions. | `COORDINATOR_PROMPT` in `agent.py`. |
| Backend | Where the agent stores files and state. | A virtual `/workspace/` route into this repo. |
| Permissions | Read/write rules for files and directories. | Writes are scoped to `llm-wiki/`; the reviewer is read-only. |
| Skills | Reusable instructions loaded for a category of work. | `research-scout`, `source-ledger`, `wiki-curator`, `aiq-research`. |
| Subagents | Focused workers with their own prompt, tools, and context. | Source scout, synthesis writer, skill router, skeptic reviewer. |

This is the frame for the rest of the notebook. Every cell below points to one of these pieces.

## 2. Start from the repo root

The notebook can be opened from the repo root or from the `notebooks/` folder. This cell finds the project root and adds `src/` to `sys.path` for local development.

In [ ]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
candidates = [start, *start.parents]
repo = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
assert repo is not None, 'Could not find repo root containing pyproject.toml'

src = repo / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print(repo)

## 3. Check the UV package surface

A workshop repo should install cleanly. `pyproject.toml` declares the package, runtime dependencies, dev dependencies, notebook dependencies, and the `deep-research-agent` console script.

In [ ]:
import tomllib

pyproject = tomllib.loads((repo / 'pyproject.toml').read_text())
project = pyproject['project']
print(project['name'], project['version'])
print('requires-python:', project['requires-python'])
print('script:', project['scripts']['deep-research-agent'])
print('runtime deps:', ', '.join(project['dependencies']))

## 4. Read the Deep Agents constructor

The constructor is the center of the project. Read it as a contract: what tools can the agent call, where can it write, which skills can it load, and which subagents can it use?

In [ ]:
import inspect

from deep_research_agent.agent import COORDINATOR_PROMPT, build_subagents, create_agent
from deep_research_agent.config import load_config, missing_model_credentials

print(inspect.getsource(create_agent))

## 5. Coordinator prompt

The prompt defines the coordinator's job before any tool call happens. It should say what artifacts matter, how evidence should be treated, and what the final response must report.

In [ ]:
print(COORDINATOR_PROMPT)

## 6. Tools are narrow contracts

Tools should be small enough to review. In this repo, the tools do deterministic work: create a task packet, normalize a source record, classify source weight, build a source ledger, and prepare an AI-Q request contract.

In [ ]:
from deep_research_agent.tools import (
    build_source_ledger,
    classify_source,
    create_task_packet,
    prepare_aiq_research_request,
    record_source,
)

packet = create_task_packet('long-running AI research agents')
packet

In [ ]:
sources = [
    record_source(
        'LangChain Deep Agents overview',
        'https://docs.langchain.com/oss/python/deepagents/overview',
        source_type='official docs',
        notes='Framework overview, built-in planning, files, subagents, memory, and skills.',
    ),
    record_source(
        'LangChain Deep Agents subagents',
        'https://docs.langchain.com/oss/python/deepagents/subagents',
        source_type='official docs',
        notes='Subagent behavior, context isolation, custom workers, and best practices.',
    ),
    record_source(
        'NVIDIA Agent Skills catalog',
        'https://github.com/NVIDIA/skills',
        source_type='repo',
        notes='Skill catalog including AI-Q skills.',
    ),
    record_source(
        'Agent Skills specification',
        'https://github.com/agentskills/agentskills',
        source_type='spec',
        notes='Portable SKILL.md package shape.',
    ),
]

for source in sources:
    weight = classify_source(source['source_type'], source['url'])['review_weight']
    print(source['source_id'], weight, source['title'])

In [ ]:
ledger = build_source_ledger('long-running AI research agents', sources)
print(ledger['markdown'])

## 7. Skills are reusable procedures

A skill is not a Python function. It is a procedure the agent can read when the task calls for that behavior. Use skills for source discipline, wiki curation, backend-specific instructions, and review expectations that should be visible to the team.

In [ ]:
def read_skill_frontmatter(path: Path) -> dict[str, str]:
    text = path.read_text()
    _, frontmatter, _ = text.split('---', 2)
    fields = {}
    for line in frontmatter.strip().splitlines():
        key, value = line.split(':', 1)
        fields[key.strip()] = value.strip()
    return fields

for skill_path in sorted((repo / '.agents' / 'skills').glob('*/SKILL.md')):
    meta = read_skill_frontmatter(skill_path)
    print(f"{meta['name']}: {meta['description']}")

## 8. Subagents keep the coordinator focused

Use subagents when a task is detailed enough to pollute the main context or specialized enough to deserve a narrower prompt. The coordinator should receive concise results from subagents, not every intermediate lookup.

In [ ]:
config = load_config()
for subagent in build_subagents(config):
    tools = [getattr(tool, '__name__', str(tool)) for tool in subagent.get('tools', [])]
    print(f"- {subagent['name']}")
    print(f"  {subagent['description']}")
    print(f"  tools: {', '.join(tools) or 'filesystem only'}")

## 9. Workspace and review artifacts

The LLM Wiki is the durable workspace. This lets the run leave behind a brief, a source ledger, open questions, and a run log. The final answer should point back to files instead of asking the reviewer to trust the conversation.

In [ ]:
workspace_files = [
    repo / 'llm-wiki' / 'README.md',
    repo / 'llm-wiki' / 'TRUST_MODEL.md',
    repo / 'llm-wiki' / 'agent' / 'TASK_PACKET_TEMPLATE.md',
    repo / 'llm-wiki' / 'wiki' / 'research-briefs' / 'TEMPLATE.md',
]

for path in workspace_files:
    print(path.relative_to(repo))

## 10. AI-Q as a skill-backed backend

AI-Q is optional in this workshop. The local helper prepares the request contract. A live backend would be handled through the `aiq-research` skill after setup. If the backend is unavailable, the agent should say so and continue with local source-ledger work.

In [ ]:
aiq_request = prepare_aiq_research_request(
    'What does NVIDIA AI-Q add to a long-running research-agent harness?',
    source_ids=[source['source_id'] for source in sources],
)
aiq_request

## 11. Write deterministic workshop artifacts

This cell creates example LLM Wiki artifacts without a live model. It gives participants something concrete to inspect before the optional agent run.

Treat these files like the first trace of the system. They show which source IDs exist, where a draft brief would land, and what a reviewer can inspect after a run.

In [ ]:
topic_slug = 'long_running_research_agents'
raw_dir = repo / 'llm-wiki' / 'raw'
brief_dir = repo / 'llm-wiki' / 'wiki' / 'research-briefs'
raw_dir.mkdir(parents=True, exist_ok=True)
brief_dir.mkdir(parents=True, exist_ok=True)

ledger_path = raw_dir / f'{topic_slug}_source_ledger.md'
brief_path = brief_dir / f'{topic_slug}.md'

ledger_path.write_text(ledger['markdown'] + '\n')
brief_path.write_text(
    '# Long-running AI research agents\n\n'
    'Status: draft for review\n\n'
    '## Summary\n\n'
    'This example brief is generated by deterministic notebook cells. '
    'A live Deep Agents run should replace this with a source-backed draft.\n\n'
    '## Source IDs\n\n'
    + '\n'.join(f"- {source['source_id']}: {source['title']}" for source in sources)
    + '\n\n## Open questions\n\n'
    + '- Which backend skills should be installed for the team environment?\n'
)

print(ledger_path.relative_to(repo))
print(brief_path.relative_to(repo))

## 12. Inspect the run

Before trusting any final answer, inspect the files. Check which sources support the central claims, which claims remain draft, which backend skills were unavailable, and which open questions still need human judgment.

For a production system, this becomes traces, evals, policy checks, and governed asset registration. For this workshop, the file tree is the trace beginners can understand.

## 13. Optional live Deep Agents run

This cell only runs when model credentials are configured. It is intentionally last so the workshop can succeed offline or in a locked-down room.

In [ ]:
from deep_research_agent.agent import invoke_task

config = load_config()
missing = missing_model_credentials(config)
if missing:
    print('Skipping live run:', missing)
else:
    result = invoke_task(
        'Compare three current approaches to long-running AI research agents. '
        'Use public sources only. Use AI-Q only if available; otherwise mark it unavailable. '
        'Stage a source-backed brief, source ledger, open questions, and run log.'
    )
    print(result)

## 14. Debrief

1. Which part of the system is framework behavior, and which part is your harness design?
2. Which files would you inspect before trusting the final answer?
3. Which claim needs a stronger source?
4. What did subagents keep out of the coordinator context?
5. Where would AI-Q or another backend skill add value?
6. Which skill would your team write next?